# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)



In [1]:
!git clone https://github.com/abbas-707/FlyRank-Internship.git
%cd FlyRank-Internship

Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 153 (delta 60), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 1.89 MiB | 8.27 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/FlyRank-Internship


In [2]:
import duckdb
from google.colab import userdata
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [3]:
model_data = con.sql("""
    WITH march_data AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id, client_hash_id
    ),
    joined AS (
        SELECT
            m.*,
            c.word_count,
            c.search_volume,
            c.competition,
            c.cpc
        FROM march_data m
        JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
            ON m.content_hash_id = c.content_hash_id
    )
    SELECT *,
        clicks_march * 1.0 / NULLIF(impressions_march, 0) AS actual_ctr,
        CASE
            WHEN avg_position_march <= 3 THEN '1. position 1-3'
            WHEN avg_position_march <= 10 THEN '2. position 4-10'
            WHEN avg_position_march <= 20 THEN '3. position 11-20'
            ELSE '4. position 20+'
        END AS position_bucket
    FROM joined
    WHERE impressions_march >= 500 AND avg_position_march > 0 AND avg_position_march <= 20
""").df()

bucket_expected = {'1. position 1-3': 0.003765, '2. position 4-10': 0.003208, '3. position 11-20': 0.002625}
model_data['expected_ctr'] = model_data['position_bucket'].map(bucket_expected)
model_data['label'] = (model_data['actual_ctr'] < model_data['expected_ctr'] * 0.7).astype(int)

numeric_features = ['impressions_march', 'avg_position_march', 'word_count', 'search_volume', 'competition', 'cpc']
model_data_clean = model_data.dropna(subset=numeric_features + ['label']).copy()

# Honest, client-grouped split (Week 5/6 design)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(model_data_clean, groups=model_data_clean['client_hash_id']))
train_df = model_data_clean.iloc[train_idx].reset_index(drop=True)
test_df = model_data_clean.iloc[test_idx].reset_index(drop=True)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[numeric_features])
X_test_scaled = scaler.transform(test_df[numeric_features])

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, train_df['label'])

test_df = test_df.copy()
test_df['model_probability'] = model.predict_proba(X_test_scaled)[:, 1]

print("Test set (used for the playbook queue):", test_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test set (used for the playbook queue): (16822, 14)


## 1. Ranked actions + reason codes

**The queue:** 16,822 pages from the held-out, client-grouped test set, ranked by the
Week 5/6 Logistic Regression's predicted probability of CTR underperformance. Each row
carries one reason code (`ctr_below_position_expectation`) and one action label
(`review_title_meta`), following the same rule established in Week 4 and validated in
Week 5-6.

**Confidence tiers:** low (<0.4, n=1,960), medium (0.4-0.6, n=14,183), high (>0.6, n=679).

**Claim, using the evidence I actually have:** the model ranks and flags pages at a
measured precision of 0.538 on this held-out, client-grouped test set (Week 6's honest
"after" number) — meaningfully better than chance, though modest, and this is a
decision-support ranking, not a diagnosis. This claim is at the "validated model that
ranks/predicts out-of-sample" tier of the claim ladder, since it's measured on data the
model never trained on.

**A named finding, not swept under the rug:** 19 of 16,822 rows (about 0.1%) have a
predicted probability of exactly 1.0 — an unusually extreme, saturated output for a
Logistic Regression. Inspecting these rows shows two patterns: (1) unusually high
`search_volume` values (33,100-60,500, well above what's typical elsewhere in this
dataset) may be dominating the linear combination after scaling; (2) 8 of these 19 rows
belong to a single client (`client_fef1a8f436438636`) — the same client that dominated
Week 4's top-10 tied-score list. Two independent weeks surfacing the same client at the
extreme end of a ranking is a real pattern worth investigating (a client-level tracking,
templating, or content-strategy difference), not something to act on as 19 independent
content problems. These 19 rows should be reviewed as a group before individual action,
not treated as the queue's most reliable top picks.

In [4]:
# Build the final ranked action queue on the test set (honest, client-grouped, out-of-sample)
test_df = test_df.copy()
test_df['reason_code'] = 'ctr_below_position_expectation'
test_df['action'] = 'review_title_meta'

# Confidence tier from the model's own probability
test_df['confidence_tier'] = pd.cut(
    test_df['model_probability'],
    bins=[0, 0.4, 0.6, 1.0],
    labels=['low', 'medium', 'high']
)

ranked_queue = test_df.sort_values('model_probability', ascending=False).reset_index(drop=True)

print("Ranked queue shape:", ranked_queue.shape)
print("\nConfidence tier distribution:")
print(ranked_queue['confidence_tier'].value_counts())
ranked_queue[['content_hash_id', 'client_hash_id', 'model_probability', 'confidence_tier',
              'reason_code', 'action']].head(10)

Ranked queue shape: (16822, 17)

Confidence tier distribution:
confidence_tier
medium    14183
low        1960
high        679
Name: count, dtype: int64


,content_hash_id,client_hash_id,model_probability,confidence_tier,reason_code,action
0,content_a85bf5efbd1f137e,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
1,content_3c8d9aaa54baddb0,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
2,content_cc9732f0da1d8d2d,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
3,content_e3bc410ce23d7400,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
4,content_99b8d4d3ed525589,client_23a62021009f63c4,1.0,high,ctr_below_position_expectation,review_title_meta
5,content_81d5735e8f392a74,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
6,content_02b62f7825be8205,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
7,content_1382f2702d83079f,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
8,content_c5dc108cc7ed608e,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta
9,content_551a522809e0358c,client_fef1a8f436438636,1.0,high,ctr_below_position_expectation,review_title_meta


## 2. Intended use and limits

**Who uses this:** a content strategist or SEO reviewer with limited weekly review
capacity, deciding which pages to look at first for a title/meta review.

**What it's for:** prioritization only — deciding where to spend a human reviewer's time
first, among pages that already have real, measured search visibility.

**Where it stops being valid:**
- Single-month snapshot (March 2026) — as noted in Week 6's leakage audit, feature and
  label windows overlap within the same month, so this describes March, not a validated
  future prediction.
- Per the `flyrank-data` skill's own warning, client history depth varies widely and a
  third of clients have little or no usable history — this playbook did not verify
  `dim_clients.gsc_data_start` per client before using March uniformly, so some clients
  in this queue may have thin or unreliable underlying history. This is a real, disclosed
  gap in this version of the work, not a silent one.
- Precision of 0.538 is modest — just over half of flagged pages are expected to be
  genuine matches; roughly the other half will not be, so this is a starting shortlist,
  not a confirmed problem list.
- The 19 saturated-probability rows (Section 1) should not be treated as the most
  trustworthy picks without a manual client-level check first.

## 3. Human review + the no-go list

**A human must check before acting on any flagged page:**
- Whether the page is technically functional (loads correctly, isn't redirecting, is
  actually indexed) — Week 4's top-10 review found every top pick had zero clicks
  despite real impressions, which can indicate a technical problem, not a copy problem.
- Whether the flagged page belongs to a client with thin underlying history (unverified
  in this version — see Section 2).
- Whether a page is one of the 19 saturated-probability rows — these need a client-level
  sanity check before being treated as individual title/meta tasks.

**What should NEVER be automated:**
- Actually rewriting or publishing new titles/meta descriptions — the model identifies
  candidates, it does not generate or approve content changes.
- Treating the ranked position within the queue as a guarantee of severity — Week 4
  showed tied/saturated scores can cluster arbitrarily at the top.
- Assuming the `review_title_meta` action is the correct fix without a human confirming
  the page isn't broken, deindexed, or a tracking artifact first.
- Applying this queue to a client not represented in the training data without re-checking
  that client's own history depth and data quality first.

## 4. Monitoring / retrain triggers

- **Data staleness:** this model is trained entirely on March 2026. It should be retrained
  or re-validated before being trusted on any later month, given the single-month scope
  disclosed in Section 2.
- **Precision drift:** if a sample of acted-on recommendations is manually reviewed and
  the real-world match rate falls meaningfully below the measured 0.538 precision, that's
  a signal the model or the underlying CTR-position relationship has shifted.
- **New saturated-probability clusters:** if future runs show a growing share of
  probability-1.0 predictions (beyond the current ~0.1%), that's a sign a feature
  (likely search_volume, per Section 1) is dominating in a way that needs investigation
  before the queue is trusted.
- **Client concentration check:** if any single client repeatedly dominates the top of
  the queue across multiple runs (as seen twice now with client_fef1a8f436438636), that's
  a trigger to investigate that client's data specifically rather than treating it as a
  content signal.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export the ranked queue (stays out of git by design)
ranked_queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print("Saved: work/outputs/action_playbook_queue.csv")

# Export the metrics JSON (this DOES get committed — it's the receipt)
import json
metrics = {
    "model": "Logistic Regression",
    "split_design": "grouped by client_hash_id, 0 overlap confirmed",
    "test_set_size": len(ranked_queue),
    "roc_auc_grouped": 0.532,
    "precision_grouped": 0.538,
    "confidence_tier_counts": ranked_queue['confidence_tier'].value_counts().to_dict(),
    "saturated_probability_rows": int((ranked_queue['model_probability'] > 0.999).sum()),
    "known_limitations": [
        "single-month snapshot (March 2026), feature/label window overlap",
        "client history depth not verified per dim_clients.gsc_data_start",
        "19 rows with saturated probability=1.0, client-concentrated"
    ]
}
with open('work/outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)
print("Saved: work/outputs/w07_playbook_metrics.json")

Saved: work/outputs/action_playbook_queue.csv
Saved: work/outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.